# 80전문가 모델: AIME 30 + LiveCodeBench 175

09·10번 노트북이 같은 소스·데이터 ZIP을 사용하지만, 이 노트북은 **80전문가 모델만**
준비하고 별도 Colab 세션에서 205문제를 각각 한 번씩 풉니다. 상대 모델 파일이나 상대
실행 결과는 준비·생성·채점·보존에 필요하지 않습니다. 모든 실행 플래그는 기본적으로 꺼져 있습니다.

`putare09-source-*.zip`과 `putare09-data-*.zip`을 `MyDrive/Putare`에 올리고, 아래 모델
경로를 준비하세요. `SETUP_COLAB`을 켜 환경을 준비한 뒤 `RUN_PREFLIGHT`, `RUN_GENERATION`,
`RUN_GRADING`, `EXPORT_RESULTS`, `DOWNLOAD_ARCHIVE`를 필요한 순서대로 켜고 실행하세요.
80전문가 GGUF는 설정된 Drive 경로에 직접 준비하세요.

RTX PRO 6000에서 입력과 출력이 공유하는 컨텍스트는 262,144토큰이며,
문제별 최대 생성량은 262,144에서 템플릿 포함 입력 토큰 수와 1토큰을 뺀 값입니다.
Thinking 활성화, Q8 KV, Top-8, routing 계측을 사용합니다. 별도 effort 단계는 없습니다.

실행 폴더는 MyDrive에 직접 저장됩니다. 중단 시 이전 완료 답변을 보존하고 아직 시작하지
않은 문제만 재개합니다. 시작했으나 답변이 없는 문제는 자동으로 다시 풀지 않습니다.
Drive 저장 공간과 연결이 부족하면 실행을 중단합니다. 토큰별 routing 기록은 매우 클 수 있습니다.

각 모델의 본 실행을 마치고 나서만 별도 실행 결과를 비교합니다. 각 세션은 같은 종류의 GPU를
사용하지만 물리적 GPU UUID는 다를 수 있으며, 속도 비교에 이 사실을 명시합니다.
채점한 생성 코드는 현재 런타임의 사용자 권한으로 실행됩니다. ZIP을 PC로 내려받은 후
`tools.comparison205.verify_archive(경로)`로 파일별 해시를 확인하세요.

비교 시 두 실행 폴더가 필요합니다. 실행 폴더 대신 ZIP만 있다면 검증한 ZIP을 풀어
각 `results` 폴더를 준비한 뒤 `comparison205.compare_runs(베이스폴더, 80전문가폴더)`를
호출할 수 있습니다. 비교할 Putare 작업 경로에도 같은 소스·데이터 ZIP이 복원되어 있어야
합니다. 이 비교 함수 자체는 GPU와 GGUF 없이 실행할 수 있습니다. 아래 노트북의 선택
비교 셀은 기존 Colab 환경 준비 후 사용합니다.


In [3]:
# 2026-11-11: 모델별로 독립된 실행 경로와 명시적 단계 플래그를 사용한다.
SETUP_COLAB = True
DOWNLOAD_MODEL = False
RUN_PREFLIGHT = True
RUN_GENERATION = True
RUN_GRADING = True
EXPORT_RESULTS = True
DOWNLOAD_ARCHIVE = False
RUN_COMPARE = False
SOURCE_ZIP = '/content/drive/MyDrive/Putare/putare09-source-b51db7ce85f8.zip'
DATA_ZIP = '/content/drive/MyDrive/Putare/putare09-data-0a3dd9112ddf.zip'
MODEL_PROFILE = 'e80'
MODEL_PATH = '/content/drive/MyDrive/models/gemma-4-26B-A4B-it-UD-IQ4_NL-lcb100-activity-ordered-e80.gguf'
RUN_DIR = '/content/drive/MyDrive/Putare/results/10-e80-205-001'
ARCHIVE_PATH = '/content/putare10-e80-results.zip'
BASE_RUN_DIR = '/content/drive/MyDrive/Putare/results/09-base205-001'
COMPARE_OUTPUT = '/content/drive/MyDrive/Putare/results/09-10-comparison-report.json'


In [4]:
if SETUP_COLAB:
    _PUTARE_SETUP_READY = False
    import hashlib, json, os, shutil, sys, tempfile, zipfile
    from pathlib import Path, PurePosixPath
    from google.colab import drive
    drive.mount('/content/drive')
    if not os.path.ismount('/content/drive'):
        raise RuntimeError('Google Drive가 마운트되지 않았습니다')
    ROOT = Path('/content/Putare') / ('b51db7ce85f806ae3c9eed3b999f11622d7eb55a16f53f6e8548e677ee958270'[:12] + '-' + '0a3dd9112ddfed70ddab7afcee7f3dae57759180a152a691175b035af5f95e7f'[:12])
    # 2026-11-11: 커널에 남은 다른 실험 버전의 모듈이 새 번들과 섞이지 않게 한다.
    for _name, _module in list(sys.modules.items()):
        if _name not in ('tools', 'moe_prune') and not _name.startswith(('tools.', 'moe_prune.')):
            continue
        _file = getattr(_module, '__file__', None)
        _paths = getattr(_module, '__path__', ()) if not _file else ()
        if (_file and not Path(_file).resolve().is_relative_to(ROOT.resolve())) or any(not Path(p).resolve().is_relative_to(ROOT.resolve()) for p in _paths):
            raise RuntimeError('다른 Putare 버전이 로드되었습니다. 커널을 다시 시작하세요: ' + _name)

    def sha256_file(path):
        with Path(path).open('rb') as stream:
            return hashlib.file_digest(stream, 'sha256').hexdigest()


    def restore_archives(root, archives):
        """모든 ZIP과 기존 경로를 먼저 검증하고 누락된 파일만 복원한다."""
        # 2026-11-09: 불완전한 전송과 경로 탈출·기존 데이터 덮어쓰기를 함께 방지한다.
        root = Path(root)
        if root.is_symlink():
            raise ValueError('복원 위치가 심볼릭 링크입니다')
        validated = []
        used = set()
        try:
            for archive_path, expected_hash in archives:
                archive_path = Path(archive_path)
                if not archive_path.is_file():
                    raise FileNotFoundError(f'Drive에 ZIP을 업로드하세요: {archive_path}')
                if sha256_file(archive_path) != expected_hash:
                    raise ValueError(f'ZIP SHA256 불일치: {archive_path}')
                archive = zipfile.ZipFile(archive_path)
                validated.append((archive, {}))
                names = archive.namelist()
                if len(names) != len(set(names)) or '_putare_inventory.json' not in names:
                    raise ValueError('ZIP 파일 목록에 중복 또는 인벤토리 누락')
                inventory = json.loads(archive.read('_putare_inventory.json'))
                if not isinstance(inventory, dict) or set(names) != set(inventory) | {'_putare_inventory.json'}:
                    raise ValueError('ZIP 인벤토리 불일치')
                for name, digest in inventory.items():
                    path = PurePosixPath(name)
                    if (not isinstance(name, str) or path.is_absolute() or not path.parts
                            or any(p in ('', '.', '..') for p in name.split('/')) or '\\' in name or ':' in name):
                        raise ValueError('안전하지 않은 ZIP 경로')
                    info = archive.getinfo(name)
                    if info.is_dir() or (info.external_attr >> 16) & 0o170000 == 0o120000:
                        raise ValueError('ZIP에 링크 또는 디렉터리가 있습니다')
                    if name in used:
                        raise ValueError('소스와 데이터 ZIP에 중복 경로가 있습니다: ' + name)
                    used.add(name)
                    with archive.open(info) as stream:
                        if hashlib.file_digest(stream, 'sha256').hexdigest() != digest:
                            raise ValueError('ZIP 파일 SHA256 불일치: ' + name)
                    target = root.joinpath(*path.parts)
                    for part in [root, *[root.joinpath(*path.parts[:i]) for i in range(1, len(path.parts) + 1)]]:
                        if part.is_symlink():
                            raise ValueError('복원 경로에 심볼릭 링크가 있습니다: ' + str(part))
                        if part != target and part.exists() and not part.is_dir():
                            raise FileExistsError('복원 폴더 위치가 파일입니다: ' + str(part))
                    if target.exists() and (not target.is_file() or sha256_file(target) != digest):
                        raise FileExistsError('기존 파일이 달라 덮어쓰지 않습니다: ' + str(target))
                    validated[-1][1][name] = target
            for archive, paths in validated:
                for name, target in paths.items():
                    if not target.exists():
                        target.parent.mkdir(parents=True, exist_ok=True)
                        with archive.open(name) as stream, target.open('xb') as output:
                            shutil.copyfileobj(stream, output)
        except (zipfile.BadZipFile, KeyError, UnicodeError) as error:
            raise ValueError('ZIP이 손상되었습니다') from error
        finally:
            for archive, _ in validated:
                archive.close()
        return len(used)

    restore_archives(ROOT, [(SOURCE_ZIP, 'b51db7ce85f806ae3c9eed3b999f11622d7eb55a16f53f6e8548e677ee958270'), (DATA_ZIP, '0a3dd9112ddfed70ddab7afcee7f3dae57759180a152a691175b035af5f95e7f')])
    os.chdir(ROOT)
    if str(ROOT) not in sys.path:
        sys.path.insert(0, str(ROOT))
    def _validate_run_directory():
        if not os.path.ismount('/content/drive'):
            raise RuntimeError('Google Drive가 연결되지 않았습니다')
        target = Path(RUN_DIR).resolve()
        drive_root = Path('/content/drive/MyDrive').resolve()
        if target == drive_root or not target.is_relative_to(drive_root):
            raise ValueError('실험 결과 경로는 MyDrive의 하위 폴더여야 합니다')
        target.mkdir(parents=True, exist_ok=True)
        with tempfile.TemporaryFile(dir=target) as probe:
            probe.write(b'putare-checkpoint')
            probe.flush()
            os.fsync(probe.fileno())
        return target
    _validate_run_directory()
    from tools.colab_setup import prepare_environment
    _PUTARE_SETUP_STATE = prepare_environment(ROOT, globals().get('_PUTARE_SETUP_STATE'))
    from tools import benchmark175, comparison205
    ENGINE = ROOT / '.cache/llama-build/bin/llama-expert-research'
    _PUTARE_SETUP_READY = True
    print('환경 준비 완료:', ROOT)


Mounted at /content/drive
ninja가 없어 설치합니다.
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition, Compute Capability: 120
공식 LCB commit 28fef95ea8c9f7a547c8329f2cd3d32b92c1fa24, venv /content/Putare/b51db7ce85f8-0a3dd9112ddf/.cache/putare-lcb/venv, source /content/Putare/b51db7ce85f8-0a3dd9112ddf/.cache/putare-lcb/livecodebench/28fef95ea8c9f7a547c8329f2cd3d32b92c1fa24
환경 준비 완료: /content/Putare/b51db7ce85f8-0a3dd9112ddf


In [5]:
if DOWNLOAD_MODEL:
    raise RuntimeError('80전문가 모델은 설정된 Drive 경로에 직접 준비하세요')


In [6]:
if RUN_PREFLIGHT:
    if not globals().get('_PUTARE_SETUP_READY', False):
        raise RuntimeError('먼저 환경 준비를 완료하세요')
    _run = _validate_run_directory()
    print('입력 및 최대 컨텍스트 점검:', comparison205.prepare_single(
        _run, model=Path(MODEL_PATH), engine=ENGINE, model_profile=MODEL_PROFILE))


입력 및 최대 컨텍스트 점검: {'ready': True, 'requests': 205, 'gpu': {'name': 'NVIDIA RTX PRO 6000 Blackwell Server Edition', 'uuid': 'GPU-cdb9b811-3b5b-98e5-8c30-09ae1f9fde79', 'total_bytes': 102641958912, 'compute_capability': '12.0', 'driver': '580.82.07'}, 'session_id': '6595721505fc41a7bf54f1b8078eeb6d'}


In [ ]:
if RUN_GENERATION:
    if not globals().get('_PUTARE_SETUP_READY', False):
        raise RuntimeError('먼저 환경 준비를 완료하세요')
    _run = _validate_run_directory()
    print('생성:', comparison205.generate_single(
        _run, model=Path(MODEL_PATH), engine=ENGINE, model_profile=MODEL_PROFILE))


e80 1/205 aime2026:1
e80 2/205 aime2026:2
e80 3/205 aime2026:3
e80 4/205 aime2026:4
e80 5/205 aime2026:5
e80 6/205 aime2026:6
e80 7/205 aime2026:7
e80 8/205 aime2026:8
e80 9/205 aime2026:9
e80 10/205 aime2026:10
e80 11/205 aime2026:11
e80 12/205 aime2026:12
e80 13/205 aime2026:13
e80 14/205 aime2026:14


In [ ]:
if RUN_GRADING:
    if not globals().get('_PUTARE_SETUP_READY', False):
        raise RuntimeError('먼저 환경 준비를 완료하세요')
    _run = _validate_run_directory()
    _report = comparison205.grade_single(
        _run, model=Path(MODEL_PATH), engine=ENGINE, model_profile=MODEL_PROFILE)
    print('완료:', _report.get('complete'), '영역별:', _report.get('domains'))
    print('문제별 결과는 실행 폴더에 보존됩니다:', _run)


In [ ]:
if EXPORT_RESULTS:
    if not globals().get('_PUTARE_SETUP_READY', False):
        raise RuntimeError('먼저 환경 준비를 완료하세요')
    _run = _validate_run_directory()
    _archive = Path(ARCHIVE_PATH)
    print('아카이브 검증:', comparison205.archive_single(_run, _archive), '경로:', _archive)


In [ ]:
if DOWNLOAD_ARCHIVE:
    if not globals().get('_PUTARE_SETUP_READY', False):
        raise RuntimeError('먼저 환경 준비를 완료하세요')
    _archive = Path(ARCHIVE_PATH)
    print('다운로드 전 검증:', comparison205.verify_archive(_archive))
    from google.colab import files
    files.download(str(_archive))


In [ ]:
if RUN_COMPARE:
    if not globals().get('_PUTARE_SETUP_READY', False):
        raise RuntimeError('먼저 환경 준비를 완료하세요')
    _run = _validate_run_directory()
    _report = comparison205.compare_runs(Path(BASE_RUN_DIR), _run, output_path=Path(COMPARE_OUTPUT))
    print('비교 완료:', _report.get('complete'), '영역별:', _report.get('domains'))
    print('문제별 비교:', _report.get('paired'), '효율:', _report.get('efficiency'))
    print('비교 결과:', COMPARE_OUTPUT)
